# Binomial model

In [59]:
import numpy as np


def european_option_binomial_model(S0, K, n, t, sigma, r, type):
    dt = t / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)

    j = np.arange(n + 1)
    S_T = S0 * u**j * d ** (n - j)  # prices init at maturity

    if type == "c":
        C = np.maximum(S_T - K, np.zeros(n + 1))  # call init at maturiy
    elif type == "p":
        C = np.maximum(K - S_T, np.zeros(n + 1))  # put init at maturiy

    for i in np.arange(n, 0, -1):
        if len(C) == 2:
            delta = (C[1] - C[0]) / (S0 * (u - d))
        C = np.exp(-r * dt) * (p * C[1 : i + 1] + (1 - p) * C[0:i])

    return C[0], delta

In [60]:
european_option_binomial_model(S0=100, K=100, n=5000, t=1, sigma=0.2, r=0.05, type="c")

(np.float64(10.450183638471676), np.float64(0.6368242698942197))

## Put-call parity

In [61]:
call = european_option_binomial_model(
    S0=100, K=100, n=5000, t=1, sigma=0.2, r=0.05, type="c"
)[0]
put = european_option_binomial_model(
    S0=100, K=100, n=5000, t=1, sigma=0.2, r=0.05, type="p"
)[0]
round(call - put, 4), round(100 - 100 * np.exp(-0.05 * 1), 4)

(np.float64(4.8771), np.float64(4.8771))

## American options

In [62]:
def american_option_binomial_model(S0, K, n, t, sigma, r, type):
    dt = t / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)

    j = np.arange(n + 1)
    S_T = S0 * u**j * d ** (n - j)

    if type == "p":
        C = np.maximum(K - S_T, np.zeros(n + 1))
    elif type == "c":
        C = np.maximum(S_T - K, np.zeros(n + 1))

    for i in np.arange(n, 0, -1):
        j_i = np.arange(i)
        S_i = S0 * u**j_i * d ** ((i - 1) - j_i)
        binomial_value = np.exp(-r * dt) * (p * C[1 : i + 1] + (1 - p) * C[0:i])
        if type == "p":
            intrinsic_value = np.maximum(K - S_i, np.zeros(i))
        elif type == "c":
            intrinsic_value = np.maximum(S_i - K, np.zeros(i))

        C = np.maximum(binomial_value, intrinsic_value)

    return C[0]

In [63]:
american_option_binomial_model(S0=100, K=100, n=5000, t=1, sigma=0.2, r=0.05, type="p")

np.float64(6.090219408091199)